# A Complete RAG Pipeline: TechNest Customer Support Assistant
## Putting Labs 5-8 together, end to end, on a brand-new knowledge base

This notebook is a **capstone project** that connects everything from the four
labs you've already worked through:

| Lab | What it taught | Where it shows up here |
|---|---|---|
| Lab 5 | Text preprocessing foundations | Step 1 |
| Lab 6 | Bag-of-Words, TF-IDF, BM25, retrieval metrics, chunking | Steps 2-4 |
| Lab 7 | Embeddings, semantic retrieval, hybrid retrieval | Steps 5-6 |
| Lab 8 | Chunk metadata, context building, prompt writing | Steps 7-8 |
| **New in this notebook** | **Actually calling an LLM to generate a grounded answer** | **Steps 9-10** |

The earlier labs stopped at *"a prompt-ready evidence package."* A **full** RAG
pipeline has one more step: sending that prompt to a real language model and
getting an answer back. That is the piece we add here.

### The big picture

```text
raw documents (NEW DATA we create below)
  -> chunking
  -> retriever (TF-IDF / BM25 / Embeddings / Hybrid)
  -> candidate evidence
  -> context building (filter outdated, dedupe, budget)
  -> prompt (weak / better / strict)
  -> LLM generation  <-- the new, final step
  -> grounded answer
```

### About the dataset

Instead of reusing the university/student-services corpus from Lab 6-8, we
build a **new** knowledge base for a fictional online electronics store,
**TechNest**. It is designed the same deliberate way the course corpora were
designed, so every failure mode you studied is reproducible:

- **Paraphrase traps** - a query says "money back", the document says "refund"
- **Exact numeric details** - prices, day windows, percentages that must not get lost by careless preprocessing
- **Outdated vs. current documents** - old shipping/warranty/return notices that conflict with today's policy
- **Multi-document dependencies** - some questions really are answered by more than one document

### Before you run this

This notebook uses `sentence-transformers`, which downloads a small model
(`all-MiniLM-L6-v2`, about 80 MB) from Hugging Face the first time you run
it, so you will need an internet connection (this works out of the box on
Google Colab). The final generation step (Step 9) can optionally call the
real Claude API - see that section for how to enable it. **Everything else
runs completely offline.**


## Step 0 - Install and import libraries

Run this once. If you're on Google Colab, just run the cell normally. If
you're local, install these into your active Python environment.


In [ ]:
# Run this cell if the packages are not already installed.
!pip install -q nltk rank-bm25 sentence-transformers scikit-learn anthropic


In [ ]:
import re
import os
import json
import string

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 100)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

print("Libraries imported and NLTK resources ready.")


## Step 1 - Create the data: the TechNest knowledge base

This is the "create data" part of the assignment. Each document is a small
Python dictionary with metadata, exactly like the corpus in Lab 8:

- `document_id` - a stable unique ID
- `title` - human-readable name of the document
- `category` - which part of the business it belongs to
- `doc_type` - policy, procedure, help page, FAQ, guide, or old notice
- `effective_date` - when the document went into effect
- `is_current` - whether this document reflects today's policy
- `text` - the actual content

Twenty of the documents are current. Three are **deliberately outdated** and
conflict with a current document, so we can test whether our pipeline
prefers up-to-date information later on.


In [ ]:
documents = [
    {"document_id": 0, "title": "Standard Shipping Policy", "category": "Shipping",
     "doc_type": "policy", "effective_date": "2025-06-01", "is_current": True,
     "text": ("Standard shipping takes 3 to 5 business days and costs 4.99 USD, "
              "free for orders over 50 USD. Express shipping takes 1 to 2 business "
              "days and costs 14.99 USD. Orders placed before 2 PM local time ship "
              "the same business day.")},

    {"document_id": 1, "title": "International Shipping Guide", "category": "Shipping",
     "doc_type": "help page", "effective_date": "2025-05-15", "is_current": True,
     "text": ("International orders take 7 to 14 business days depending on the "
              "destination country. Customers are responsible for customs duties "
              "and import taxes, which are calculated at checkout for supported "
              "countries. Some products cannot be shipped internationally because "
              "of battery restrictions.")},

    {"document_id": 2, "title": "Order Tracking and Delivery Issues", "category": "Orders",
     "doc_type": "help page", "effective_date": "2025-07-01", "is_current": True,
     "text": ("Tracking numbers are emailed within 24 hours of shipment. If a "
              "package shows as delivered but was not received, contact support "
              "within 5 days so an investigation can begin. Packages that stay "
              "in transit more than 10 business days past the estimated delivery "
              "date are eligible for reshipment or a refund.")},

    {"document_id": 3, "title": "Return and Refund Policy", "category": "Returns & Refunds",
     "doc_type": "policy", "effective_date": "2025-06-10", "is_current": True,
     "text": ("Most items can be returned within 30 days of delivery for a full "
              "refund, provided the item is unused and in its original packaging. "
              "Opened software, personal care items, and gift cards are final "
              "sale. Refunds are issued to the original payment method within 5 "
              "to 7 business days after the returned item is received and "
              "inspected.")},

    {"document_id": 4, "title": "How Refunds Are Processed", "category": "Returns & Refunds",
     "doc_type": "procedure", "effective_date": "2025-06-10", "is_current": True,
     "text": ("Once a return arrives at the warehouse it is inspected within 2 "
              "business days. Approved refunds are issued to the original payment "
              "method, while store-credit refunds are issued instantly. Refunds "
              "for orders originally paid with a gift card are issued as store "
              "credit only, never as cash back.")},

    {"document_id": 5, "title": "Order Cancellation Policy", "category": "Orders",
     "doc_type": "policy", "effective_date": "2025-06-20", "is_current": True,
     "text": ("Orders can be cancelled for free within 1 hour of purchase from "
              "the order history page. After 1 hour, if the order has not yet "
              "shipped, customers may request cancellation through support, but "
              "it is not guaranteed. Orders that have already shipped cannot be "
              "cancelled and must instead be returned after delivery under the "
              "standard return policy.")},

    {"document_id": 6, "title": "Standard Warranty Coverage", "category": "Warranty & Repairs",
     "doc_type": "policy", "effective_date": "2025-01-01", "is_current": True,
     "text": ("All TechNest products purchased in 2022 or later include a 1 year "
              "manufacturer warranty covering defects in materials and "
              "workmanship. The warranty does not cover accidental damage, water "
              "damage, or unauthorized repairs. Proof of purchase is required for "
              "every warranty claim.")},

    {"document_id": 7, "title": "Extended Protection Plan", "category": "Warranty & Repairs",
     "doc_type": "policy", "effective_date": "2025-02-01", "is_current": True,
     "text": ("Customers may purchase an Extended Protection Plan at checkout for "
              "2 additional years of coverage beyond the standard warranty, for a "
              "fee equal to roughly 10 to 15 percent of the item price. The plan "
              "also covers one accidental damage incident.")},

    {"document_id": 8, "title": "Filing a Warranty Claim", "category": "Warranty & Repairs",
     "doc_type": "procedure", "effective_date": "2025-02-15", "is_current": True,
     "text": ("To file a warranty claim, submit the product serial number, proof "
              "of purchase, and a description of the defect through the support "
              "portal. Approved claims receive a prepaid return shipping label. "
              "Repairs typically take 10 to 15 business days; if a repair cannot "
              "be completed, a replacement or refund is issued instead.")},

    {"document_id": 9, "title": "Payment Methods Accepted", "category": "Payments & Billing",
     "doc_type": "FAQ", "effective_date": "2025-03-01", "is_current": True,
     "text": ("TechNest accepts major credit and debit cards, PayPal, and TechNest "
              "gift cards. Buy now pay later financing is available on orders "
              "over 100 USD through a third party provider. Payment information "
              "is encrypted and never stored on TechNest servers.")},

    {"document_id": 10, "title": "Billing Errors and Duplicate Charges", "category": "Payments & Billing",
     "doc_type": "help page", "effective_date": "2025-03-10", "is_current": True,
     "text": ("A duplicate charge is usually a temporary authorization hold that "
              "disappears within 3 to 5 business days. If a real duplicate "
              "charge remains after 5 business days, contact billing support "
              "with the order number so it can be reversed.")},

    {"document_id": 11, "title": "Password Reset for Customer Accounts", "category": "Account & Security",
     "doc_type": "procedure", "effective_date": "2025-04-01", "is_current": True,
     "text": ("Customers can reset a forgotten password using the Forgot "
              "Password link on the sign in page, which emails a reset link to "
              "the registered address. The reset link expires after 30 minutes. "
              "If the registered email is no longer accessible, customers must "
              "verify identity with order details through support.")},

    {"document_id": 12, "title": "Two-Factor Authentication Setup", "category": "Account & Security",
     "doc_type": "help page", "effective_date": "2025-04-05", "is_current": True,
     "text": ("Two factor authentication adds a verification code sent by text "
              "message or an authenticator app during sign in. It can be enabled "
              "from the account security settings page and is strongly "
              "recommended for accounts with saved payment methods.")},

    {"document_id": 13, "title": "TechNest Rewards Program", "category": "Rewards Program",
     "doc_type": "FAQ", "effective_date": "2025-01-15", "is_current": True,
     "text": ("Rewards members earn 2 points per dollar spent, redeemable for "
              "discounts starting at 500 points for 5 USD off. Points expire "
              "after 12 months of account inactivity. Membership is free and "
              "enrollment happens automatically with a customer's first "
              "purchase.")},

    {"document_id": 14, "title": "Gift Card Terms", "category": "Gift Cards",
     "doc_type": "policy", "effective_date": "2025-01-20", "is_current": True,
     "text": ("TechNest gift cards never expire and carry no maintenance fees. "
              "Gift cards cannot be redeemed for cash except where required by "
              "law. A lost gift card code can be recovered by contacting support "
              "with proof of purchase.")},

    {"document_id": 15, "title": "Discount Codes and Promotions", "category": "Promotions",
     "doc_type": "FAQ", "effective_date": "2025-05-01", "is_current": True,
     "text": ("Discount codes can be applied at checkout and cannot be combined "
              "with other promotional codes. Only one discount code is allowed "
              "per order. Promotional pricing is not applied retroactively to "
              "orders placed before a promotion started.")},

    {"document_id": 16, "title": "Headphones Pairing Troubleshooting", "category": "Product Support",
     "doc_type": "guide", "effective_date": "2025-05-20", "is_current": True,
     "text": ("If wireless headphones will not pair, reset them by holding the "
              "power button for 10 seconds until the light flashes, then forget "
              "the device in the phone's Bluetooth settings before pairing "
              "again. Keep the headphones within 3 feet of the device during "
              "pairing.")},

    {"document_id": 17, "title": "Laptop Won't Turn On Troubleshooting", "category": "Product Support",
     "doc_type": "guide", "effective_date": "2025-05-25", "is_current": True,
     "text": ("If a laptop will not power on, hold the power button for 15 "
              "seconds to perform a hard reset, then charge it for at least 30 "
              "minutes before trying again. If the charging light never turns "
              "on, the charger or battery likely needs warranty service.")},

    {"document_id": 18, "title": "International Customs and Import Duties", "category": "International Orders",
     "doc_type": "help page", "effective_date": "2025-05-15", "is_current": True,
     "text": ("Import duties and taxes for international orders are calculated "
              "at checkout for supported countries and are non-refundable once "
              "an order ships, even if the order is later returned. Refused "
              "international shipments are subject to a restocking fee equal to "
              "the return shipping cost.")},

    {"document_id": 19, "title": "Contacting Customer Support", "category": "Customer Support",
     "doc_type": "FAQ", "effective_date": "2025-06-01", "is_current": True,
     "text": ("Support is available by live chat and email 7 days a week from 8 "
              "AM to 10 PM local time. Phone support is available Monday through "
              "Friday from 9 AM to 6 PM. Average email response time is under 4 "
              "hours during business hours.")},

    # --- Outdated documents that conflict with current policy ---
    {"document_id": 20, "title": "Return Policy Notice (Effective 2023)", "category": "Returns & Refunds",
     "doc_type": "old notice", "effective_date": "2023-01-10", "is_current": False,
     "text": ("Effective for purchases made before January 2024, the return "
              "window was 14 days from delivery instead of the current 30 day "
              "window. This notice is retained for historical reference only "
              "and no longer applies.")},

    {"document_id": 21, "title": "Old Standard Shipping Rates (2022)", "category": "Shipping",
     "doc_type": "old notice", "effective_date": "2022-03-01", "is_current": False,
     "text": ("Before the 2023 shipping update, standard shipping cost 7.99 USD "
              "with no free shipping threshold, and delivery took 5 to 7 "
              "business days. These rates no longer apply to current orders.")},

    {"document_id": 22, "title": "Legacy Warranty Terms (2021)", "category": "Warranty & Repairs",
     "doc_type": "old notice", "effective_date": "2021-01-01", "is_current": False,
     "text": ("Products purchased before 2022 carried a 90 day manufacturer "
              "warranty instead of the current 1 year warranty. This notice is "
              "retained for record purposes only and does not apply to current "
              "purchases.")},
]

documents_df = pd.DataFrame(documents)
print("Number of documents:", len(documents_df))
print("Current documents:", int(documents_df["is_current"].sum()))
print("Outdated documents:", int((~documents_df["is_current"]).sum()))
documents_df[["document_id", "title", "category", "doc_type", "effective_date", "is_current"]]


### Why this corpus creates real retrieval pressure

| Pressure type | Example |
|---|---|
| Paraphrase | `money back` vs `refund` |
| Exact detail | prices, day windows, percentages |
| Outdated vs current | 2022 shipping rates vs today's rates |
| Multi-document | gift-card refunds depend on both the refund policy and the gift-card terms |


## Step 1b - Queries and ground truth

A query is what a customer might type. Ground truth is the document ID (or
IDs) that a human says is actually relevant. Several queries below use
**different words** than the document they map to — that's on purpose, and
it's what will separate lexical retrieval from semantic retrieval later.

We also keep two **out-of-scope** queries aside. They have no relevant
document at all, and we'll use them later to test whether the pipeline knows
how to say "I don't know" instead of making something up.


In [ ]:
ground_truth = {
    # ── Tier A: Paraphrase / semantic queries ─────────────────────────────────
    # These use DIFFERENT words than the relevant document — testing whether
    # embeddings bridge the vocabulary gap that breaks TF-IDF / BM25.
    "How can I get my money back for something I returned?": [3, 4],
    "How many days do I have to send something back?": [3],
    "My order says delivered but it never showed up": [2],
    "Can I cancel my order after I already paid for it?": [5],
    "I forgot my password and can\'t log in": [11],
    "How do I turn on extra security for my account?": [12],
    "My wireless headphones won\'t connect to my phone": [16],
    "My laptop won\'t start, what should I do?": [17],
    "Do gift cards expire?": [14],
    "Can I combine two discount codes on one order?": [15],
    "What are your customer support hours?": [19],
    "If I paid with a gift card and return the item, do I get cash back?": [4],
    "What happens if a package is stuck in transit past the delivery date?": [2],

    # ── Tier B: Numeric-detail queries (deliberate embedding weakness) ─────────
    # These queries hinge on EXACT numbers (prices, day counts, percentages).
    # A sentence-embedding model assigns nearly the same vector to
    # "4.99 USD" and "7.99 USD" because they encode the same semantic role
    # (a shipping price). BM25 treats them as different token sequences and
    # is therefore more precise on these queries.
    # Use these to show where hybrid retrieval beats pure semantic search.
    "How much does standard shipping cost right now?": [0],           # $4.99 (old doc says $7.99)
    "What is the cutoff time to get same-day shipping?": [0],          # 2 PM — precision matters
    "What is the current warranty length on a new TechNest purchase?": [6],  # 1 year (old: 90 days)
    "How many extra years does the Extended Protection Plan add?": [7],  # exactly 2 years
    "How long does a warranty repair typically take?": [8],             # 10-15 business days
    "How many points do I need to get 5 dollars off?": [13],           # 500 points (old: 750)
    "After how many months of inactivity do my rewards points expire?": [13],  # 12 months (old: 6)
    "I see two identical charges for the same order, when will they resolve?": [10],  # 3-5 days
    "How long does the password reset link stay valid?": [11],          # exactly 30 minutes
    "What percentage volume discount do I get for 10+ units?": [22],   # 5% — number must match

    # ── Tier C: Multi-document queries ────────────────────────────────────────
    # Correct answer requires evidence from MORE THAN ONE document.
    # Tests recall and context-building breadth.
    "Do I get my customs fees back if I return an international order?": [18],
    "What happens to a warranty claim if the repair fails?": [8],
    "What\'s the free shipping minimum right now?": [0],
}

out_of_scope_queries = [
    "Do you sell reusable water bottles?",
    "Can I get a refund in cryptocurrency?",
    "Is TechNest publicly traded on the stock market?",
    "Do you have physical retail stores?",
]

queries_df = pd.DataFrame({
    "query": list(ground_truth.keys()),
    "relevant_document_ids": list(ground_truth.values()),
    "is_numeric_detail": [
        any(c.isdigit() for c in q) for q in ground_truth.keys()
    ],
})
print(f"Total queries: {len(queries_df)}")
print(f"Numeric-detail queries: {queries_df['is_numeric_detail'].sum()}")
print(f"Semantic queries: {(~queries_df['is_numeric_detail']).sum()}")
queries_df


## Step 1c - Save the data as standalone files

The assignment asked us to "create data" — so in addition to using it inline
in this notebook, let's export it as plain JSON files you could hand to any
other retrieval system, notebook, or teammate.


In [ ]:
os.makedirs("technest_data", exist_ok=True)

with open("technest_data/knowledge_base.json", "w") as f:
    json.dump(documents, f, indent=2)

with open("technest_data/queries_ground_truth.json", "w") as f:
    json.dump(
        {"ground_truth": ground_truth, "out_of_scope_queries": out_of_scope_queries},
        f, indent=2
    )

print("Saved technest_data/knowledge_base.json")
print("Saved technest_data/queries_ground_truth.json")


## Step 2 - Text preprocessing refresher (Lab 5)

Before we vectorize anything, remember the central lesson of Lab 5:
**preprocessing is a set of tradeoffs, not a checklist to maximize.** Let's
prove it again on our own data before moving on, so the lesson carries over
instead of being forgotten.

Watch what happens to the exact numbers in the shipping sentence below once
we turn on aggressive cleaning.


In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(
    text,
    lowercase=True,
    remove_punct=True,
    remove_num=False,
    remove_stop_words=False,
    preserve_negation=True,
    use_stemming=False,
    use_lemmatization=False,
):
    if use_stemming and use_lemmatization:
        raise ValueError("Use either stemming or lemmatization, not both at the same time.")

    if lowercase:
        text = text.lower()
    if remove_punct:
        text = text.translate(str.maketrans("", "", string.punctuation))
    if remove_num:
        text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return ""

    tokens = word_tokenize(text)

    if remove_stop_words:
        protected_negation_words = {"no", "not", "nor", "never"}
        if preserve_negation:
            tokens = [t for t in tokens if (t not in stop_words) or (t in protected_negation_words)]
        else:
            tokens = [t for t in tokens if t not in stop_words]

    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
    if use_lemmatization:
        tokens = [lemmatizer.lemmatize(t, pos="v") for t in tokens]

    return " ".join(tokens)


preprocessing_profiles = {
    "minimal_clean": dict(remove_stop_words=False, use_stemming=False, use_lemmatization=False),
    "stopword_reduced": dict(remove_stop_words=True, preserve_negation=True),
    "aggressive_stemmed": dict(remove_stop_words=True, preserve_negation=False, remove_num=True, use_stemming=True),
    "readable_lemmatized": dict(remove_stop_words=True, preserve_negation=True, use_lemmatization=True),
}

sample_texts = [
    "My order says DELIVERED but I never got it!!!",
    "Can I cancel my order after I already paid?",
    "The return window is NOT the same as the warranty period.",
    "Standard shipping costs 4.99 dollars for 3-5 business days.",
]

rows = []
for text in sample_texts:
    row = {"raw_text": text}
    for name, kwargs in preprocessing_profiles.items():
        row[name] = preprocess_text(text, **kwargs)
    rows.append(row)

pd.DataFrame(rows)


**The lesson repeats itself on new data:** `aggressive_stemmed` mangles
"4.99 dollars" and "3-5 business days" into digit-free mush, which would be
disastrous for a support bot that needs to quote exact prices and windows.
For **retrieval**, we will therefore keep numbers and use only light,
reversible cleaning (lowercase + basic punctuation normalization) — the same
choice Lab 6 and Lab 8 made.


## Step 3 - Chunk the documents (Lab 6 / Lab 8)

Each document is short, but in a real knowledge base documents can be much
longer, and the answer to a query often lives in just one part of one
document. We split every document into overlapping word chunks, and we carry
the full metadata onto every chunk so we can still filter by
`is_current`, `effective_date`, etc. later.

We also build a `search_text` field that prepends the title and category to
the chunk text, exactly like Lab 8 did — this helps both TF-IDF and
embeddings use document-level context, not just the raw words of the chunk.


In [ ]:
def chunk_text(text, chunk_size=40, overlap=10):
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0:
        raise ValueError("overlap cannot be negative")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start += chunk_size - overlap
    return chunks


chunk_rows = []
for doc in documents:
    for chunk_index, chunk_value in enumerate(chunk_text(doc["text"])):
        chunk_rows.append({
            "chunk_id": f"doc{doc['document_id']}_chunk{chunk_index}",
            "document_id": doc["document_id"],
            "title": doc["title"],
            "category": doc["category"],
            "doc_type": doc["doc_type"],
            "effective_date": doc["effective_date"],
            "is_current": doc["is_current"],
            "chunk_index": chunk_index,
            "chunk_text": chunk_value,
            "search_text": f"{doc['title']} {doc['category']} {doc['doc_type']} {chunk_value}",
        })

chunks_df = pd.DataFrame(chunk_rows)
print("Number of source documents:", len(documents_df))
print("Number of chunks:", len(chunks_df))
chunks_df.head(6)


## Step 4 - Lexical retrieval: Bag-of-Words, TF-IDF, and BM25 (Lab 6)

### 4.1 A quick Bag-of-Words look

Bag-of-Words keeps only which words appear and how many times, ignoring
grammar and order entirely. It's the simplest possible representation, and
it's useful for building intuition before jumping to TF-IDF.


In [ ]:
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(chunks_df["chunk_text"])
vocabulary = count_vectorizer.get_feature_names_out()

print("Vocabulary size:", len(vocabulary))
print("Matrix shape (chunks x vocabulary):", bow_matrix.shape)

total_values = bow_matrix.shape[0] * bow_matrix.shape[1]
non_zero_values = bow_matrix.count_nonzero()
sparsity_percentage = (total_values - non_zero_values) / total_values * 100
print(f"Sparsity: {sparsity_percentage:.1f}% of the matrix is zero")


### 4.2 TF-IDF retriever

TF-IDF upweights words that are frequent in one chunk but rare across the
whole corpus, so generic words contribute less than specific ones. We
normalize the text lightly first (lowercase, strip punctuation, keep
numbers) and use unigrams **and** bigrams so two-word phrases like
`password reset` are captured as a single feature.


In [ ]:
def normalize_lexical_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(chunks_df["search_text"].map(normalize_lexical_text))

print("TF-IDF matrix shape:", tfidf_matrix.shape)


def retrieve_top_k_tfidf(query, k=3):
    query_vector = tfidf_vectorizer.transform([normalize_lexical_text(query)])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    ranking = np.argsort(scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["score"] = scores[ranking]
    results["retriever"] = "TF-IDF"
    return results[["retriever", "chunk_id", "document_id", "title",
                     "effective_date", "is_current", "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_tfidf("How can I get my money back for something I returned?", k=5)


### 4.3 BM25 retriever

BM25 is still lexical (word-overlap based), but it adds term-frequency
saturation and document-length normalization, which usually makes it a
stronger keyword baseline than raw TF-IDF.


In [ ]:
def simple_tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


tokenized_chunks = [simple_tokenize(text) for text in chunks_df["search_text"]]
bm25 = BM25Okapi(tokenized_chunks)


def retrieve_top_k_bm25(query, k=3):
    scores = bm25.get_scores(simple_tokenize(query))
    ranking = np.argsort(scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["score"] = scores[ranking]
    results["retriever"] = "BM25"
    return results[["retriever", "chunk_id", "document_id", "title",
                     "effective_date", "is_current", "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_bm25("How can I get my money back for something I returned?", k=5)


## Step 5 - Retrieval metrics (Lab 6 / 7 / 8)

We evaluate every retriever with the same four metrics so comparisons are
fair:

| Metric | Question it answers |
|---|---|
| Precision@K | Of the top K results, how many are relevant? |
| Recall@K | Of all relevant documents, how many did we find in the top K? |
| Hit Rate@K | Did we find at least one relevant document in the top K? |
| Reciprocal Rank (-> MRR) | How early did the first relevant result appear? |


In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return len(hits) / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return len(hits) / len(relevant_ids)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return int(len(hits) > 0)


def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in set(relevant_ids):
            return 1 / rank
    return 0.0


def evaluate_retriever(retriever_name, retrieval_function, ground_truth, k=3):
    rows = []
    for query, relevant_ids in ground_truth.items():
        results = retrieval_function(query, k)
        retrieved_doc_ids = results["document_id"].tolist()
        rows.append({
            "retriever": retriever_name,
            "query": query,
            "relevant_ids": relevant_ids,
            "retrieved_doc_ids": retrieved_doc_ids,
            f"precision@{k}": precision_at_k(retrieved_doc_ids, relevant_ids, k),
            f"recall@{k}": recall_at_k(retrieved_doc_ids, relevant_ids, k),
            f"hit_rate@{k}": hit_rate_at_k(retrieved_doc_ids, relevant_ids, k),
            "reciprocal_rank": reciprocal_rank(retrieved_doc_ids, relevant_ids),
        })
    return pd.DataFrame(rows)


K = 3
tfidf_eval = evaluate_retriever("TF-IDF", retrieve_top_k_tfidf, ground_truth, k=K)
bm25_eval = evaluate_retriever("BM25", retrieve_top_k_bm25, ground_truth, k=K)

lexical_summary = pd.concat([tfidf_eval, bm25_eval], ignore_index=True).groupby("retriever")[
    [f"precision@{K}", f"recall@{K}", f"hit_rate@{K}", "reciprocal_rank"]
].mean()
lexical_summary


### A lexical failure, on purpose

Let's look directly at the paraphrase trap query. The relevant documents (3
and 4) talk about "refund", but the query says "money back".


In [ ]:
retrieve_top_k_tfidf("How can I get my money back for something I returned?", k=5)[
    ["document_id", "title", "score"]
]


If the refund documents aren't sitting clearly at the top, that's the
vocabulary-mismatch problem Lab 7 warned about — not a bug. It's exactly why
we add semantic embeddings next.


## Step 6 - Semantic retrieval with embeddings (Lab 7)

An embedding is a dense numerical vector that represents *meaning* instead of
exact words. We use `all-MiniLM-L6-v2`, a small pretrained sentence-embedding
model, so "money back" and "refund" can land close together in vector space
even though they share no words.

**The first time you run this cell, it downloads the model from Hugging
Face** — you need an internet connection (Colab has one by default).


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(
    chunks_df["search_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Chunk embeddings shape:", chunk_embeddings.shape)


In [ ]:
def retrieve_top_k_semantic(query, k=3):
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()
    ranking = np.argsort(scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["score"] = scores[ranking]
    results["retriever"] = "Embeddings"
    return results[["retriever", "chunk_id", "document_id", "title",
                     "effective_date", "is_current", "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_semantic("How can I get my money back for something I returned?", k=5)


This should surface the refund and refund-processing documents even though
the words don't overlap. That's the "aha" moment of semantic search.

Now let's put a number on it, comparing all three retrievers so far.


In [ ]:
embedding_eval = evaluate_retriever("Embeddings", retrieve_top_k_semantic, ground_truth, k=K)

summary_so_far = pd.concat(
    [tfidf_eval, bm25_eval, embedding_eval], ignore_index=True
).groupby("retriever")[
    [f"precision@{K}", f"recall@{K}", f"hit_rate@{K}", "reciprocal_rank"]
].mean().sort_values(by="reciprocal_rank", ascending=False)

summary_so_far


## Step 7 - Hybrid retrieval (Lab 7 / 8)

Neither approach wins every time:

- BM25/TF-IDF are strong on exact words, prices, and codes.
- Embeddings are strong on paraphrases and meaning, but can blur exact
  numeric details (recall the "refund within 7 days" vs "refund within 14
  days" failure case from Lab 7).

Hybrid retrieval blends both signals:

$$\text{hybrid score} = \alpha \times \text{semantic score} + (1-\alpha) \times \text{lexical score}$$

Because BM25 and cosine-similarity scores live on different scales, we
min-max normalize both to [0, 1] before combining them.


In [ ]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=float)
    lo, hi = scores.min(), scores.max()
    if hi == lo:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)


def retrieve_top_k_hybrid(query, alpha=0.6, k=3):
    lexical_query_vector = tfidf_vectorizer.transform([normalize_lexical_text(query)])
    lexical_scores = cosine_similarity(lexical_query_vector, tfidf_matrix).flatten()

    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    semantic_scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()

    combined_scores = (
        alpha * min_max_normalize(semantic_scores)
        + (1 - alpha) * min_max_normalize(lexical_scores)
    )
    ranking = np.argsort(combined_scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["tfidf_score"] = lexical_scores[ranking]
    results["semantic_score"] = semantic_scores[ranking]
    results["score"] = combined_scores[ranking]
    results["retriever"] = f"Hybrid alpha={alpha}"
    return results[["retriever", "chunk_id", "document_id", "title",
                     "effective_date", "is_current", "tfidf_score",
                     "semantic_score", "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_hybrid("How can I get my money back for something I returned?", alpha=0.6, k=5)


In [ ]:
hybrid_evals = []
for alpha in [0.2, 0.5, 0.8]:
    ev = evaluate_retriever(
        retriever_name=f"Hybrid alpha={alpha}",
        retrieval_function=lambda q, k, a=alpha: retrieve_top_k_hybrid(q, alpha=a, k=k),
        ground_truth=ground_truth,
        k=K,
    )
    hybrid_evals.append(ev)

final_summary = pd.concat(
    [tfidf_eval, bm25_eval, embedding_eval] + hybrid_evals, ignore_index=True
).groupby("retriever")[
    [f"precision@{K}", f"recall@{K}", f"hit_rate@{K}", "reciprocal_rank"]
].mean().sort_values(by="reciprocal_rank", ascending=False)

final_summary


There's no universally correct alpha — pick the one with the best
Reciprocal Rank / Hit Rate on **your own** validation queries, the same way
Lab 7 and Lab 8 taught. We'll use the best-performing alpha found above as
our default retriever for the rest of the notebook.


In [ ]:
# Pick the best-performing hybrid alpha specifically (by Reciprocal Rank),
# so later steps have one clear default weight to use.
hybrid_rows = final_summary[final_summary.index.str.startswith("Hybrid")]
best_hybrid_name = hybrid_rows["reciprocal_rank"].idxmax()
BEST_ALPHA = float(best_hybrid_name.split("=")[-1])
print("Best-performing hybrid alpha:", BEST_ALPHA)
print("Using alpha =", BEST_ALPHA, "as the default hybrid weight going forward.")


## Step 8 - Context building (Lab 8)

Retrieval gives us **candidate** evidence. Context building decides what the
model actually sees. Good context is:

| Property | Why it matters |
|---|---|
| Relevant | Only chunks related to the query |
| Current | Prefer `is_current=True` sources over outdated ones |
| Non-redundant | No duplicate chunks from the same document |
| Compact | Stays within a word budget |
| Labeled | Title, date, and CURRENT/OUTDATED marker included |


In [ ]:
def build_context_package(
    query,
    retrieval_k=8,
    alpha=BEST_ALPHA,
    max_context_chunks=3,
    max_chunks_per_document=1,
    word_budget=150,
    prefer_current=True,
    min_score_ratio=0.40,
    min_absolute_score=0.0,
):
    candidates = retrieve_top_k_hybrid(query, alpha=alpha, k=retrieval_k).copy()

    if prefer_current:
        candidates = candidates.sort_values(
            by=["is_current", "score", "effective_date"],
            ascending=[False, False, False],
        ).reset_index(drop=True)

    max_score = candidates["score"].max() if len(candidates) else 0.0
    selected_rows = []
    seen_texts = set()
    per_document_counts = {}
    used_words = 0

    for _, row in candidates.iterrows():
        if row["score"] < min_absolute_score:
            continue
        if max_score > 0 and row["score"] < max_score * min_score_ratio:
            continue

        normalized = re.sub(r"\s+", " ", row["chunk_text"]).strip().lower()
        if normalized in seen_texts:
            continue

        doc_count = per_document_counts.get(row["document_id"], 0)
        if doc_count >= max_chunks_per_document:
            continue

        chunk_words = len(row["chunk_text"].split())
        if selected_rows and used_words + chunk_words > word_budget:
            continue

        selected_rows.append(row.to_dict())
        seen_texts.add(normalized)
        per_document_counts[row["document_id"]] = doc_count + 1
        used_words += chunk_words

        if len(selected_rows) >= max_context_chunks:
            break

    blocks = []
    for position, row in enumerate(selected_rows, start=1):
        currency_label = "CURRENT" if row["is_current"] else "OUTDATED"
        blocks.append(
            f"[Source {position}] {row['title']} | {row['effective_date']} | {currency_label}\n"
            f"{row['chunk_text']}"
        )

    return {
        "query": query,
        "candidates": candidates,
        "selected_df": pd.DataFrame(selected_rows),
        "context_text": "\n\n".join(blocks),
        "used_words": used_words,
        "num_sources": len(selected_rows),
    }


sample_package = build_context_package("How much does shipping cost right now?", retrieval_k=10)
print(f"Word budget used: {sample_package['used_words']} | sources selected: {sample_package['num_sources']}")
print()
print(sample_package["context_text"])


### A current-vs-outdated conflict, handled

The shipping query above could have surfaced the 2022 outdated rates
document (id 21) alongside the current one (id 0). Because
`build_context_package` sorts by `is_current` first, the outdated document is
pushed down (or dropped entirely by the per-document / score-ratio filters)
instead of confusing the final answer.


## Step 9 - Prompt writing (Lab 8)

The prompt controls *how* the model uses the evidence: whether it stays
grounded, cites sources, handles conflicts, or refuses when the context isn't
enough. We build the same three prompt styles Lab 8 introduced.


In [ ]:
def build_weak_prompt(query, context_text):
    return f"""Answer the question using the context.

Question:
{query}

Context:
{context_text}
"""


def build_better_prompt(query, context_text):
    return f"""You are a careful TechNest customer support assistant.

Answer using only the provided context.

Rules:
1. Do not use outside knowledge.
2. If the context is not enough to answer, say so clearly.
3. If sources disagree, prefer the most current source and mention the conflict.
4. Cite the source numbers you use in your answer.
5. Keep the answer concise but complete.

Question:
{query}

Context:
{context_text}
"""


def build_strict_prompt(query, context_text):
    return f"""You are a grounded RAG assistant for TechNest customer support.

Rules:
1. Use only the provided context. Never add background knowledge.
2. If the answer is not in the context, say: "The provided sources do not contain enough information to answer this question."
3. If a source is marked OUTDATED, do not use it as the primary answer. Mention it only to note the conflict.
4. If current and outdated sources conflict, state the conflict and use the CURRENT source.
5. Output exactly two sections:
   Answer: [your grounded answer]
   Sources: [list the source numbers you used]

Question:
{query}

Context:
{context_text}
"""


PROMPT_BUILDERS = {"weak": build_weak_prompt, "better": build_better_prompt, "strict": build_strict_prompt}

print(build_strict_prompt(sample_package["query"], sample_package["context_text"]))


## Step 10 - Generation: actually calling an LLM (new step)

This is the piece Lab 8 stopped short of. Everything up to now produced a
prompt-ready package of evidence. A **full** RAG pipeline sends that prompt
to a real language model and returns the model's answer.

### Getting an API key (optional but recommended)

To make real calls, you need your own Claude API key:

1. Create a key at [console.anthropic.com](https://console.anthropic.com)
2. Set it as an environment variable before running this cell, e.g. in a
   Colab cell: `import os; os.environ["ANTHROPIC_API_KEY"] = "your-key-here"`
   (or use Colab's "Secrets" panel so it isn't stored in the notebook file)

**No key? No problem.** The `generate_answer` function below falls back to a
simple offline mode that just surfaces the retrieved evidence, so the whole
pipeline still runs end to end while you decide whether to add a key.


In [ ]:
def fallback_extractive_answer(query, context_text):
    """
    Used only when no API key is configured. It doesn't reason about the
    question - it just surfaces the retrieved evidence, so the pipeline still
    produces something useful without requiring a paid API call.
    """
    if not context_text.strip():
        return ("Answer: The provided sources do not contain enough information "
                "to answer this question.\nSources: none")
    return (
        "Answer (offline fallback - no LLM call made; set ANTHROPIC_API_KEY to "
        f"enable real generation):\n\n{context_text}\n\n"
        "Sources: see the bracketed [Source N] labels above."
    )


def generate_answer(query, context_text, prompt_style="better", model_name="claude-sonnet-5", max_tokens=600):
    prompt = PROMPT_BUILDERS[prompt_style](query, context_text)

    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return fallback_extractive_answer(query, context_text)

    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
    response = client.messages.create(
        model=model_name,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return "".join(block.text for block in response.content if block.type == "text")


print(generate_answer(sample_package["query"], sample_package["context_text"], prompt_style="strict"))


## Step 11 - Putting it all together: `answer_question()`

Now we wrap retrieval, context building, prompting, and generation into a
single function — this is the complete RAG pipeline.


In [ ]:
def answer_question(query, prompt_style="better", alpha=BEST_ALPHA, verbose=True):
    package = build_context_package(query, retrieval_k=10, alpha=alpha)
    answer = generate_answer(query, package["context_text"], prompt_style=prompt_style)

    if verbose:
        print("QUERY:", query)
        print("-" * 70)
        print(f"Sources used ({package['num_sources']}, {package['used_words']} words):")
        print(package["context_text"] if package["context_text"] else "(no sources cleared the relevance filters)")
        print("-" * 70)
        print("ANSWER:")
        print(answer)
        print("=" * 70)

    return {"query": query, "context_package": package, "answer": answer}


demo_queries = [
    "How can I get my money back for something I returned?",
    "How much does shipping cost right now?",
    "My laptop won't start, what should I do?",
]

for q in demo_queries:
    answer_question(q, prompt_style="strict")


### Testing the "I don't know" behavior

A trustworthy support assistant should say it doesn't know rather than
invent an answer. Let's try our two out-of-scope queries.


In [ ]:
for q in out_of_scope_queries:
    answer_question(q, prompt_style="strict")


## Step 12 - Full-pipeline evaluation and error analysis

Metrics tell us *how much* it worked. Error analysis tells us *why*. Let's
run every ground-truth query through the retrieval layer one more time (with
our chosen hybrid alpha) and flag any that still miss, so we know exactly
where to look if the final generated answer ever looks wrong.


In [ ]:
final_hybrid_eval = evaluate_retriever(
    retriever_name=f"Hybrid alpha={BEST_ALPHA}",
    retrieval_function=lambda q, k: retrieve_top_k_hybrid(q, alpha=BEST_ALPHA, k=k),
    ground_truth=ground_truth,
    k=K,
)

failed_queries = final_hybrid_eval[final_hybrid_eval[f"hit_rate@{K}"] == 0]
print(f"{len(failed_queries)} of {len(final_hybrid_eval)} queries missed at top-{K}.")
failed_queries[["query", "relevant_ids", "retrieved_doc_ids"]]


For each failed query, the same checklist from Lab 6/7/8 applies:

1. What was the query, and what should have been retrieved?
2. What was retrieved instead?
3. Was it a vocabulary mismatch (fixable by leaning more on semantic score),
   an exact-detail mismatch (fixable by leaning more on lexical score), or a
   genuinely ambiguous / multi-document question?
4. Would a different `alpha`, a larger `retrieval_k`, or better chunking fix it?

Retrieval failures propagate forward: no prompt or model can rescue a
question whose correct document was never retrieved in the first place.


## Summary and takeaways

**Data (Step 1)**
- We built a 23-document TechNest knowledge base with the same structure Lab
  8 used: metadata, paraphrase traps, exact numeric details, and outdated
  documents that conflict with current policy.

**Retrieval (Steps 2-7)**
- Preprocessing is a tradeoff, not a checklist — light, reversible cleaning
  wins for retrieval.
- TF-IDF and BM25 are strong on exact words, weak on paraphrases.
- Embeddings are strong on meaning, weaker on exact numbers/codes.
- Hybrid retrieval combined both signals and was evaluated across several
  alpha values rather than assumed to be best.

**Context and prompting (Steps 8-9)**
- Retrieved chunks are candidate evidence, not final context — filtering
  outdated sources, deduplicating, and respecting a word budget all matter.
- Prompts are control instructions: weak prompts invite hallucination, strict
  prompts force grounding and citations.

**Generation (Step 10) - the new piece**
- A full RAG pipeline ends with an actual model call, not just a prompt.
- A working fallback means the pipeline still runs and teaches the concept
  even without a paid API key.

**The pipeline, once more**
```text
retrieval quality -> context quality -> answer quality
```
A failure at any layer propagates forward. Fix the layer, not the symptom.

## Suggested exercises

1. Add 5 more documents and 5 more queries of your own, including at least
   one new paraphrase trap and one new outdated-vs-current conflict.
2. Try `max_chunks_per_document=2` in `build_context_package` and see how it
   changes the shipping-query context.
3. Compare the `weak`, `better`, and `strict` prompts side by side for the
   printing-cost-style outdated/current conflict query.
4. If you have an API key, compare the real generated answer for a
   paraphrase-trap query against the offline fallback, and note the
   difference.
5. Swap `all-MiniLM-L6-v2` for a different sentence-transformers model and
   re-run the evaluation — did Hit Rate@3 or MRR change?
